# Weekly Product Demand Forecast

We forecast weekly units for each product at each branch using three approaches: last week, a four-week average, and a trained linear regression. The goal is to compare simple forecasting methods, not to create an automatic order recommendation. The sales data is synthetic.


## The plan

- **Baseline 1:** predict next week will match the previous week.
- **Baseline 2:** predict next week using the average of the previous four weeks.
- **Learned model:** train a linear regression on past weekly sales, recent sales lags, week of year, branch, and product.
- **Time split:** use the earlier 80% of weeks as history and reserve the latest 20% for validation. We do not shuffle weeks.

Each method predicts one week ahead. After a validation week has finished, its actual sales become available as past data for the next week.

## 1. Load the prepared sales

Run the data_preparation notebook first so the cleaned CSV exists.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'data' / 'processed' / 'clean_sales_2025.csv').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

SALES_FILE = ROOT / 'data' / 'processed' / 'clean_sales_2025.csv'
if not SALES_FILE.exists():
    raise FileNotFoundError('Run the Milestone 2 notebook first to create the cleaned sales file.')

sales = pd.read_csv(SALES_FILE, parse_dates=['sale_date'])
REPORTS_DIR = ROOT / 'reports'
print(f'Product lines loaded: {len(sales):,}')
print(f'Sales dates: {sales.sale_date.min():%d %b %Y} to {sales.sale_date.max():%d %b %Y}')


Product lines loaded: 1,511
Sales dates: 01 Jan 2025 to 31 Dec 2025


## 2. Total units by product, branch, and week

The source file only lists weeks when a product sold. We add zero-sales weeks too, so a quiet week is counted as zero rather than disappearing from the forecast.

In [2]:
group_columns = ['store_id', 'store_name', 'product_id', 'product_name']

# Start each week on Monday.
sales['week_start'] = sales['sale_date'] - pd.to_timedelta(sales['sale_date'].dt.weekday, unit='D')
weekly_totals = (
    sales.groupby(group_columns + ['week_start'], as_index=False)
    .agg(units_sold=('quantity', 'sum'))
)

all_weeks = pd.date_range(weekly_totals['week_start'].min(), weekly_totals['week_start'].max(), freq='7D')
week_table = pd.DataFrame({'week_start': all_weeks})
product_branches = weekly_totals[group_columns].drop_duplicates()

# Make every product-branch combination have a row for every week.
weekly = product_branches.merge(week_table, how='cross')
weekly = weekly.merge(weekly_totals, on=group_columns + ['week_start'], how='left')
weekly['units_sold'] = weekly['units_sold'].fillna(0).astype(int)
weekly = weekly.sort_values(group_columns + ['week_start']).reset_index(drop=True)

print(f'Product-branch combinations: {len(product_branches)}')
print(f'Weeks in the timeline: {len(all_weeks)}')
print(f'Weekly rows, including zero-sales weeks: {len(weekly):,}')
weekly.head()

Product-branch combinations: 20
Weeks in the timeline: 53
Weekly rows, including zero-sales weeks: 1,060


  store_id            store_name  ... week_start units_sold
0     S001  Sample CBD Mini-Mart  ... 2024-12-30          4
1     S001  Sample CBD Mini-Mart  ... 2025-01-06          3
2     S001  Sample CBD Mini-Mart  ... 2025-01-13          4
3     S001  Sample CBD Mini-Mart  ... 2025-01-20          1
4     S001  Sample CBD Mini-Mart  ... 2025-01-27          6

[5 rows x 6 columns]

## 3. Create the time-based validation split

The first 80% of weeks are history. We reserve the latest 20% for validation. The model learns only from the history weeks. We do not shuffle weeks. Each validation prediction uses actual sales from earlier completed weeks, just as a one-week-ahead forecast would.


In [3]:
week_list = weekly['week_start'].drop_duplicates().sort_values().tolist()
split_number = int(len(week_list) * 0.80)
training_weeks = week_list[:split_number]
validation_weeks = week_list[split_number:]

print(f'History weeks: {len(training_weeks)}')
print(f'Validation weeks: {len(validation_weeks)}')
print(f'Validation period: {validation_weeks[0]:%d %b %Y} to {validation_weeks[-1]:%d %b %Y}')

# For each product and branch, these columns contain sales from earlier weeks only.
for number_of_weeks_ago in range(1, 5):
    weekly[f'week_{number_of_weeks_ago}_ago'] = (
        weekly.groupby(group_columns)['units_sold'].shift(number_of_weeks_ago)
    )

weekly['baseline_forecast'] = weekly['week_1_ago']
past_week_columns = [f'week_{number}_ago' for number in range(1, 5)]
weekly['four_week_average_forecast'] = weekly[past_week_columns].mean(axis=1)

# Week of year is represented as a repeating calendar pattern.
week_number = weekly['week_start'].dt.isocalendar().week.astype(int)
weekly['week_sin'] = np.sin(2 * np.pi * week_number / 52.18)
weekly['week_cos'] = np.cos(2 * np.pi * week_number / 52.18)


History weeks: 42
Validation weeks: 11
Validation period: 20 Oct 2025 to 29 Dec 2025


## 4. Train the linear regression model

Linear regression learns weights from earlier examples. Here it uses the previous four weeks of sales, their average, the time of year, and labels for the branch and product. The branch and product labels are turned into simple 0/1 columns. NumPy finds the weights that best fit the training data. We set predictions below zero to zero because sales units cannot be negative.

This is one shared model across all products and branches. The first four weeks are needed to build its past-sales features, so they are not training rows.

In [4]:
numeric_features = ['week_1_ago', 'week_2_ago', 'week_3_ago', 'week_4_ago', 'four_week_average_forecast', 'week_sin', 'week_cos']
category_features = pd.get_dummies(
    weekly[['store_id', 'product_id']].astype('string'),
    columns=['store_id', 'product_id'],
    drop_first=True,
    dtype=float,
)
model_features = pd.concat([weekly[numeric_features], category_features], axis=1).astype(float)
feature_ready = weekly[numeric_features].notna().all(axis=1)

training_mask = weekly['week_start'].isin(training_weeks) & feature_ready
validation_mask = weekly['week_start'].isin(validation_weeks) & feature_ready

training_design = np.column_stack([np.ones(training_mask.sum()), model_features.loc[training_mask].to_numpy()])
training_target = weekly.loc[training_mask, 'units_sold'].to_numpy(dtype=float)
linear_coefficients, *_ = np.linalg.lstsq(training_design, training_target, rcond=None)

validation = weekly.loc[validation_mask].copy()
validation_design = np.column_stack([np.ones(validation_mask.sum()), model_features.loc[validation_mask].to_numpy()])
validation['linear_regression_forecast'] = np.maximum(0, validation_design @ linear_coefficients)

print(f'Training rows used: {training_mask.sum()}')
print(f'Validation product-branch-weeks: {validation_mask.sum()}')
validation[['week_start', 'store_name', 'product_name', 'units_sold', 'baseline_forecast', 'four_week_average_forecast', 'linear_regression_forecast']].head()


Training rows used: 760
Validation product-branch-weeks: 220


   week_start  ... linear_regression_forecast
42 2025-10-20  ...                   3.922027
43 2025-10-27  ...                   3.773073
44 2025-11-03  ...                   3.794550
45 2025-11-10  ...                   3.798715
46 2025-11-17  ...                   3.807328

[5 rows x 7 columns]

## 5. Compare forecasts with actual sales

- **MAE** is the average number of units missed. Lower is better.
- **WAPE** is the total absolute miss divided by total actual units. We use it instead of MAPE because some weeks have zero sales.
- **Bias** is the average forecast minus actual units. Positive means over-forecasting; negative means under-forecasting.

All three methods use the same held-out weeks, products, and branches.

In [5]:
forecast_columns = {
    'Last-week baseline': 'baseline_forecast',
    'Four-week average baseline': 'four_week_average_forecast',
    'Learned linear regression': 'linear_regression_forecast',
}
evaluation_rows = []
for forecast_name, forecast_column in forecast_columns.items():
    error = validation[forecast_column] - validation['units_sold']
    validation[f"{forecast_column}_absolute_error"] = error.abs()
    evaluation_rows.append({
        'forecast': forecast_name,
        'MAE_units': error.abs().mean(),
        'WAPE_pct': error.abs().sum() / validation['units_sold'].sum() * 100,
        'bias_units': error.mean(),
    })
evaluation = pd.DataFrame(evaluation_rows).round(2)
evaluation


                     forecast  MAE_units  WAPE_pct  bias_units
0          Last-week baseline       2.20    104.53        0.15
1  Four-week average baseline       1.73     82.00        0.12
2   Learned linear regression       1.54     72.91        0.09

## 6. What the learned model gets wrong

The largest misses show where the regression still struggles. In this sample, recent-sales features smooth out sudden jumps and can stay too high after a quiet week.

In [6]:
validation['linear_regression_error'] = validation['linear_regression_forecast'] - validation['units_sold']
validation['miss_direction'] = validation['linear_regression_error'].apply(
    lambda error: 'over-forecast' if error > 0 else ('under-forecast' if error < 0 else 'exact')
)
largest_misses = validation.sort_values('linear_regression_forecast_absolute_error', ascending=False)[
    ['week_start', 'store_name', 'product_name', 'units_sold', 'linear_regression_forecast', 'miss_direction', 'linear_regression_forecast_absolute_error']
].head(5).copy()
largest_misses[['linear_regression_forecast', 'linear_regression_forecast_absolute_error']] = largest_misses[['linear_regression_forecast', 'linear_regression_forecast_absolute_error']].round(2)
largest_misses


    week_start  ... linear_regression_forecast_absolute_error
581 2025-12-22  ...                                      6.46
574 2025-11-03  ...                                      6.25
891 2025-10-27  ...                                      5.67
210 2025-12-22  ...                                      5.36
367 2025-12-08  ...                                      4.83

[5 rows x 7 columns]

## 7. Conclusion and saved results

A lower score is better, but even the best result may still have too much error for a stock decision. We also refit the regression on all available history for a historical one-week-ahead example.

In [7]:
baseline_mae = evaluation.loc[evaluation['forecast'].eq('Last-week baseline'), 'MAE_units'].iloc[0]
average_mae = evaluation.loc[evaluation['forecast'].eq('Four-week average baseline'), 'MAE_units'].iloc[0]
linear_mae = evaluation.loc[evaluation['forecast'].eq('Learned linear regression'), 'MAE_units'].iloc[0]
baseline_wape = evaluation.loc[evaluation['forecast'].eq('Last-week baseline'), 'WAPE_pct'].iloc[0]
average_wape = evaluation.loc[evaluation['forecast'].eq('Four-week average baseline'), 'WAPE_pct'].iloc[0]
linear_wape = evaluation.loc[evaluation['forecast'].eq('Learned linear regression'), 'WAPE_pct'].iloc[0]

if linear_mae < average_mae:
    mae_improvement = (average_mae - linear_mae) / average_mae * 100
    conclusion = (
        f'The trained linear regression beat the four-week average on the same validation weeks. '        f'MAE fell from {average_mae:.2f} to {linear_mae:.2f} units ({mae_improvement:.1f}% lower). '        f'WAPE fell from {average_wape:.1f}% to {linear_wape:.1f}%. '        'However, the remaining error is still high for restocking.'
    )
else:
    conclusion = (
        f'The trained linear regression did not beat the four-week average on the same validation weeks. '        f'Its MAE was {linear_mae:.2f} units versus {average_mae:.2f}, and its WAPE was {linear_wape:.1f}% versus {average_wape:.1f}%. '        'The remaining error is still high for restocking.'
    )
print(conclusion)

# Fit the same regression approach on all available history for a historical example.
all_history_mask = feature_ready
all_history_design = np.column_stack([np.ones(all_history_mask.sum()), model_features.loc[all_history_mask].to_numpy()])
all_history_target = weekly.loc[all_history_mask, 'units_sold'].to_numpy(dtype=float)
all_history_coefficients, *_ = np.linalg.lstsq(all_history_design, all_history_target, rcond=None)

last_observed_week = weekly['week_start'].max()
forecast_week_start = last_observed_week + pd.Timedelta(days=7)
next_week_rows = weekly.loc[weekly['week_start'].eq(last_observed_week)].copy()
next_week_rows['week_start'] = forecast_week_start
next_week_rows['baseline_forecast'] = next_week_rows['units_sold']
next_week_rows['week_4_ago'] = next_week_rows['week_3_ago']
next_week_rows['week_3_ago'] = next_week_rows['week_2_ago']
next_week_rows['week_2_ago'] = next_week_rows['week_1_ago']
next_week_rows['week_1_ago'] = next_week_rows['units_sold']
next_week_rows['four_week_average_forecast'] = next_week_rows[past_week_columns].mean(axis=1)
next_week_number = next_week_rows['week_start'].dt.isocalendar().week.astype(int)
next_week_rows['week_sin'] = np.sin(2 * np.pi * next_week_number / 52.18)
next_week_rows['week_cos'] = np.cos(2 * np.pi * next_week_number / 52.18)
next_category_features = pd.get_dummies(
    next_week_rows[['store_id', 'product_id']].astype('string'),
    columns=['store_id', 'product_id'],
    drop_first=True,
    dtype=float,
)
next_numeric = next_week_rows[numeric_features]
next_features = pd.concat([next_numeric, next_category_features], axis=1).reindex(columns=model_features.columns, fill_value=0).astype(float)
next_design = np.column_stack([np.ones(len(next_features)), next_features.to_numpy()])
next_week_rows['linear_regression_forecast'] = np.maximum(0, next_design @ all_history_coefficients)
next_week_rows['forecast_week_start'] = forecast_week_start
next_week_rows['forecast_units'] = next_week_rows['four_week_average_forecast']
next_week_forecast = next_week_rows[group_columns + ['forecast_week_start', 'baseline_forecast', 'four_week_average_forecast', 'linear_regression_forecast', 'forecast_units']].copy()
next_week_forecast = next_week_forecast.rename(columns={'baseline_forecast': 'last_week_baseline_units'})
next_week_forecast.to_csv(REPORTS_DIR / 'next_week_forecast.csv', index=False)
print(f'Historical example forecast saved for week starting {forecast_week_start:%d %b %Y}.')

worst_misses_text = largest_misses.to_string(index=False)
last_validation_week = max(validation_weeks)
last_week_actual = validation.loc[validation['week_start'].eq(last_validation_week), 'units_sold'].sum()
last_week_model = validation.loc[validation['week_start'].eq(last_validation_week), 'linear_regression_forecast'].sum()
report_lines = [
    '# Demand Forecast Evaluation',
    '',
    'This forecast uses synthetic weekly sales data. It is a learning example, not an order recommendation for a real shop.',
    '',
    '## Forecast setup',
    '',
    f'Training/history: {len(training_weeks)} earlier weeks ({training_mask.sum()} product-branch-week rows used for training after lag features). Validation: {len(validation_weeks)} later weeks, from {validation_weeks[0]:%d %b %Y} to {validation_weeks[-1]:%d %b %Y}.',
    'Each method predicts one week ahead. The validation week that has finished is available as past data for the next prediction.',
    'The learned model is one linear regression using four sales lags, the average of those lags, a repeating week-of-year pattern, and branch/product indicators. Negative predictions are set to zero.',
    '',
    '## Validation metrics on the same 11 weeks',
    '',
    '| Forecast | MAE (units) | WAPE | Bias (units) |',
    '| --- | ---: | ---: | ---: |',
    *[f"| {row['forecast']} | {row['MAE_units']:.2f} | {row['WAPE_pct']:.1f}% | {row['bias_units']:.2f} |" for _, row in evaluation.iterrows()],
    '',
    '## Conclusion',
    '',
    conclusion,
    'WAPE of the learned model is still high: on these validation rows, the total absolute error is about 72.9% of actual units sold. Use the result to compare methods, not to set real shop orders.',
    '',
    '## What the learned model gets wrong',
    '',
    f'The largest misses include sudden demand spikes. For example, the Ruiru branch sold 10 units of maize flour 2kg in the weeks starting 3 November and 22 December, while the regression predicted about 3.75 and 3.54 units. In the final validation week, total actual sales were {last_week_actual:.0f} units versus {last_week_model:.1f} predicted units, so the model also stayed too high after demand fell.',
    '',
    'Largest individual product-branch-week misses:',
    '',
    '```text',
    worst_misses_text,
    '```',
    '',
    '## Historical post-sample forecast',
    '',
    f'The saved example is for the week starting {forecast_week_start:%d %b %Y}, immediately after the 2025 sample. It is historical now. The CSV includes both baselines and the learned regression forecast.',
    'A real restocking model should be checked against actual point-of-sale and stockout data over more history before anyone uses it for orders.',
]

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
(REPORTS_DIR / 'evaluation.md').write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
validation_output_columns = ['week_start', 'store_name', 'product_name', 'units_sold', 'baseline_forecast', 'four_week_average_forecast', 'linear_regression_forecast', 'baseline_forecast_absolute_error', 'four_week_average_forecast_absolute_error', 'linear_regression_forecast_absolute_error']
validation[validation_output_columns].to_csv(REPORTS_DIR / 'validation_predictions.csv', index=False)
print('Saved the evaluation report, validation predictions, and historical forecast in reports/')


The trained linear regression beat the four-week average on the same validation weeks. MAE fell from 1.73 to 1.54 units (11.0% lower). WAPE fell from 82.0% to 72.9%. However, the remaining error is still high for restocking.
Historical example forecast saved for week starting 05 Jan 2026.
Saved the evaluation report, validation predictions, and historical forecast in reports/
